# 05. Insights Summary
Tổng hợp insights từ EDA để chuẩn bị cho dashboard và presentation.

In [10]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [11]:
df = pd.read_csv('../data/processed/combined_tiki_data.csv')
df['discount_pct'] = ((df['original_price'] - df['price']) / df['original_price'] * 100).round(2)
df['revenue_estimate'] = df['price'] * df['quantity_sold']
print(f"Dataset: {df.shape}")

Dataset: (41576, 21)


## Consolidated Insights

In [12]:
insights = {
    'overview': {
        'total_products': int(len(df)),
        'total_categories': int(df['category'].nunique()),
        'total_brands': int(df['brand'].nunique()),
        'total_revenue_estimate': float(df['revenue_estimate'].sum()),
        'total_quantity_sold': int(df['quantity_sold'].sum()),
        'avg_price': float(df['price'].mean()),
        'avg_rating': float(df['rating_average'].mean())
    },
    'price_segments': {},
    'top_performers': {},
    'market_structure': {},
    'customer_engagement': {},
    'key_findings': []
}

print("Overview:")
for key, value in insights['overview'].items():
    print(f"  {key}: {value}")

Overview:
  total_products: 41576
  total_categories: 6
  total_brands: 824
  total_revenue_estimate: 86397708452.0
  total_quantity_sold: 739094
  avg_price: 272358.878271118
  avg_rating: 1.3815446411391188


## Price Segment Insights

In [13]:
bins = [0, 100000, 300000, 500000, 1000000, float('inf')]
labels = ['0-100k', '100-300k', '300-500k', '500k-1M', '>1M']
df['price_segment'] = pd.cut(df['price'], bins=bins, labels=labels)

segment_stats = df.groupby('price_segment').agg({
    'id': 'count',
    'revenue_estimate': 'sum',
    'quantity_sold': 'mean'
})

for segment in labels:
    insights['price_segments'][segment] = {
        'product_count': int(segment_stats.loc[segment, 'id']),
        'product_pct': float(segment_stats.loc[segment, 'id'] / len(df) * 100),
        'revenue': float(segment_stats.loc[segment, 'revenue_estimate']),
        'revenue_pct': float(segment_stats.loc[segment, 'revenue_estimate'] / df['revenue_estimate'].sum() * 100),
        'avg_sales': float(segment_stats.loc[segment, 'quantity_sold'])
    }

dominant_segment = segment_stats['id'].idxmax()
insights['key_findings'].append({
    'title': 'Price Segment Dominance',
    'finding': f"{dominant_segment} segment contains {insights['price_segments'][dominant_segment]['product_pct']:.1f}% of products but generates only {insights['price_segments'][dominant_segment]['revenue_pct']:.1f}% of revenue",
    'implication': 'Opportunity to focus on higher-margin segments'
})

print(f"Dominant segment: {dominant_segment}")

Dominant segment: 0-100k


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19108\1576920791.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment_stats = df.groupby('price_segment').agg({


## Top Performers

In [14]:
top_products = df.nlargest(10, 'revenue_estimate')
top_categories = df.groupby('category')['revenue_estimate'].sum().nlargest(3)
top_brands = df.groupby('brand')['revenue_estimate'].sum().nlargest(10)

insights['top_performers'] = {
    'top_products': top_products[['name', 'brand', 'price', 'quantity_sold', 'revenue_estimate']].to_dict('records'),
    'top_categories': {cat: float(rev) for cat, rev in top_categories.items()},
    'top_brands': {brand: float(rev) for brand, rev in top_brands.items()}
}

top_100_revenue = df.nlargest(100, 'revenue_estimate')['revenue_estimate'].sum()
revenue_concentration = top_100_revenue / df['revenue_estimate'].sum() * 100

insights['key_findings'].append({
    'title': 'Revenue Concentration',
    'finding': f"Top 100 products (0.24% of catalog) generate {revenue_concentration:.1f}% of total revenue",
    'implication': 'Highly concentrated market - focus on top performers'
})

print(f"Top 3 categories: {list(top_categories.index)}")

Top 3 categories: ['backpacks_suitcases', 'men_shoes', 'fashion_accessories']


## Market Structure

In [15]:
oem_count = (df['brand'] == 'OEM').sum()
oem_revenue = df[df['brand'] == 'OEM']['revenue_estimate'].sum()
branded_avg_price = df[df['brand'] != 'OEM']['price'].mean()
oem_avg_price = df[df['brand'] == 'OEM']['price'].mean()

insights['market_structure'] = {
    'oem_product_count': int(oem_count),
    'oem_pct': float(oem_count / len(df) * 100),
    'oem_revenue_pct': float(oem_revenue / df['revenue_estimate'].sum() * 100),
    'branded_avg_price': float(branded_avg_price),
    'oem_avg_price': float(oem_avg_price),
    'price_premium': float(branded_avg_price / oem_avg_price)
}

insights['key_findings'].append({
    'title': 'OEM vs Branded',
    'finding': f"OEM products are {insights['market_structure']['oem_pct']:.1f}% of catalog, but branded products command {insights['market_structure']['price_premium']:.1f}x price premium",
    'implication': 'Brand value is significant in this market'
})

print(f"OEM percentage: {insights['market_structure']['oem_pct']:.1f}%")

OEM percentage: 73.8%


## Customer Engagement

In [16]:
correlation = df['review_count'].corr(df['quantity_sold'])

high_reviews = df[df['review_count'] >= 50]['quantity_sold'].mean()
low_reviews = df[df['review_count'] < 10]['quantity_sold'].mean()
review_multiplier = high_reviews / low_reviews if low_reviews > 0 else 0

insights['customer_engagement'] = {
    'review_sales_correlation': float(correlation),
    'review_impact_multiplier': float(review_multiplier),
    'avg_rating': float(df['rating_average'].mean()),
    'products_with_reviews': int((df['review_count'] > 0).sum()),
    'review_coverage_pct': float((df['review_count'] > 0).sum() / len(df) * 100)
}

insights['key_findings'].append({
    'title': 'Review Impact on Sales',
    'finding': f"Products with 50+ reviews sell {review_multiplier:.1f}x more than products with <10 reviews (correlation: {correlation:.3f})",
    'implication': 'Review generation is critical for sales success'
})

print(f"Review-sales correlation: {correlation:.3f}")

Review-sales correlation: 0.621


## Save Insights

In [17]:
output_path = Path('../data/processed')
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / 'insights_summary.json', 'w', encoding='utf-8') as f:
    json.dump(insights, f, indent=2, ensure_ascii=False)

print(f"Insights saved to {output_path / 'insights_summary.json'}")
print(f"\nTotal key findings: {len(insights['key_findings'])}")

Insights saved to ..\data\processed\insights_summary.json

Total key findings: 4


## Summary

In [18]:
print("\n=== KEY FINDINGS SUMMARY ===")
for i, finding in enumerate(insights['key_findings'], 1):
    print(f"\n{i}. {finding['title']}")
    print(f"   Finding: {finding['finding']}")
    print(f"   Implication: {finding['implication']}")


=== KEY FINDINGS SUMMARY ===

1. Price Segment Dominance
   Finding: 0-100k segment contains 55.7% of products but generates only 28.1% of revenue
   Implication: Opportunity to focus on higher-margin segments

2. Revenue Concentration
   Finding: Top 100 products (0.24% of catalog) generate 31.8% of total revenue
   Implication: Highly concentrated market - focus on top performers

3. OEM vs Branded
   Finding: OEM products are 73.8% of catalog, but branded products command 4.1x price premium
   Implication: Brand value is significant in this market

4. Review Impact on Sales
   Finding: Products with 50+ reviews sell 147.4x more than products with <10 reviews (correlation: 0.621)
   Implication: Review generation is critical for sales success
